In [7]:
# Memuat API key dari file .env
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
# Mendefinisikan tool kustom (buatan sendiri) menggunakan decorator @tool
# - Nama tool dan deskripsinya sangat penting agar LLM paham kapan harus memanggil tool ini
from langchain.tools import tool

# Database tiruan kelas pelatihan di Pusdiklat
COURSES_DB = {
    "dasar ai": "Kelas 'Dasar-Dasar AI untuk PNS' (ID: AI-101) berlangsung selama 3 hari dengan instruktur Budi Santoso. Sisa kuota: 5 peserta.",
    "langchain": "Kelas 'Penerapan LangChain di Pemerintahan' (ID: LC-202) berlangsung selama 5 hari dengan instruktur Siti Rahma. Sisa kuota: PENUH (0 peserta).",
    "keamanan data": "Kelas 'Keamanan Siber & Perlindungan Data' (ID: SEC-303) berlangsung selama 4 hari dengan instruktur Andi Wijaya. Sisa kuota: 12 peserta."
}

@tool("get_course_info", description="Mendapatkan informasi detail kelas pelatihan di Pusdiklat berdasarkan nama topik/kelas.")
def get_course_info(topic: str) -> str:
    topic_lower = topic.lower()
    for key, info in COURSES_DB.items():
        if key in topic_lower or topic_lower in key:
            return info
    return f"Maaf, kelas pelatihan dengan topik '{topic}' tidak ditemukan di database Pusdiklat."


In [9]:
# Membuat agent baru dengan mendaftarkan custom tool yang telah dibuat
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_course_info],
    system_prompt="Anda adalah asisten virtual Pusdiklat. Bantu pengguna untuk menjawab pertanyaan seputar kelas pelatihan menggunakan tool yang tersedia. Selalu gunakan tool untuk mencari info kelas sebelum menjawab."
)

In [10]:
# Mengirim pertanyaan tentang informasi kelas pelatihan Pusdiklat
from langchain.messages import HumanMessage

question = HumanMessage(content="Apakah kelas LangChain di Pusdiklat masih buka? Siapa pengajarnya?")

response = agent.invoke(
    {"messages": [question]}
)

# Menampilkan jawaban akhir LLM
last_message = response['messages'][-1]
if isinstance(last_message.content, list) and len(last_message.content) > 0:
    print(last_message.content[0]['text'])
else:
    print(last_message.content)


Mohon maaf, saat ini kelas **"Penerapan LangChain di Pemerintahan" (ID: LC-202)** sudah **PENUH** (kuota 0 peserta), sehingga pendaftaran sudah ditutup.

Kelas ini dipandu oleh instruktur **Siti Rahma** dan berlangsung selama 5 hari. Jika Anda membutuhkan informasi mengenai kelas lain atau jadwal berikutnya, silakan beri tahu saya.


In [11]:
# Menampilkan seluruh riwayat pertukaran pesan (pesan user, pemanggilan tool, hasil tool, dan jawaban LLM)
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Apakah kelas LangChain di Pusdiklat masih buka? Siapa pengajarnya?', additional_kwargs={}, response_metadata={}, id='ce29c268-ff91-4571-a97f-580d1f4290b4'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_course_info', 'arguments': '{"topic": "LangChain"}'}, '__gemini_function_call_thought_signatures__': {'jiKqCaYb': 'EjQKMgERTTIPJ2ij/zjdidoVTgmPnP2ATU8vp+rIlg15GEVE4HCM1zDtrAUNjd9qV/axziSs'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f6dde-bbc5-7301-a4b6-3fb47c8a4d58-0', tool_calls=[{'name': 'get_course_info', 'args': {'topic': 'LangChain'}, 'id': 'jiKqCaYb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 19, 'total_tokens': 127, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="Kelas 'Penerapan LangChain di Pemerintahan' (ID: LC-202) berlangsung selama 5 har

In [12]:
# Melihat detail pemanggilan tool (tool_calls) yang dieksekusi oleh LLM pada pesan ke-2
print(response["messages"][1].tool_calls)

[{'name': 'get_course_info', 'args': {'topic': 'LangChain'}, 'id': 'jiKqCaYb', 'type': 'tool_call'}]
